In [1]:
import scipy
import netCDF4
import xarray as xr
import s3fs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


/home/arunavk/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
shot = 30421
base = f"s3://mast/level2/shots/{shot}.zarr"

fs = s3fs.S3FileSystem(
    anon=True,
    client_kwargs={"endpoint_url": "https://s3.echo.stfc.ac.uk"},
)

mapper = fs.get_mapper(base)

# Try consolidated=True first (faster). If it fails, set consolidated=False.
eq = xr.open_zarr(mapper, group="equilibrium", consolidated=True)
pf = xr.open_zarr(mapper, group="pf_active", consolidated=True)

eq


<xarray.Dataset> Size: 10MB
Dimensions:              (time: 146, z: 65, major_radius: 65,
                          n_boundary_coords: 139, n_x_points_r: 2,
                          n_x_points_z: 2, profile_r: 65)
Coordinates:
  * major_radius         (major_radius) float64 520B 0.06 0.09 ... 1.95 1.98
  * n_boundary_coords    (n_boundary_coords) float32 556B 0.0 1.0 ... 138.0
  * n_x_points_r         (n_x_points_r) <U16 128B 'EFM_XPOINT1_R(C)' 'EFM_XPO...
  * n_x_points_z         (n_x_points_z) <U16 128B 'EFM_XPOINT1_Z(C)' 'EFM_XPO...
  * profile_r            (profile_r) float32 260B 0.0 0.01562 ... 0.9844 1.0
  * time                 (time) float64 1kB -0.0612 -0.0562 ... 0.6588 0.6638
  * z                    (z) float32 260B -2.0 -1.938 -1.875 ... 1.875 1.938 2.0
Data variables: (12/35)
    beta_pol             (time) float64 1kB ...
    beta_tor             (time) float64 1kB ...
    beta_tor_normal      (time) float64 1kB ...
    bphi_rmag            (time) float64 1kB ...
    bvac_rmag            (time) float64 1kB ...
    da_rating            (time) float64 1kB ...
    ...                   ...
    triangularity_upper  (time) float64 1kB ...
    vloop_dynamic        (time) float64 1kB ...
    vloop_static         (time) float64 1kB ...
    whmd                 (time) float64 1kB ...
    x_point_r            (n_x_points_r, time) float64 2kB ...
    x_point_z            (n_x_points_z, time) float64 2kB ...
Attributes:
    description:  Description of a 2D, axi-symmetric, tokamak equilibrium; re...
    label:        q(r) at z=0.
    name:         equilibrium
    imas:         equilibrium
    units:        
    license:      {'name': 'Creative Commons 4.0 BY-SA', 'url': 'https://crea...

In [ ]:
import zarr

root = zarr.open_group(mapper, mode="r")
print("Groups:", list(root.group_keys()))
print("Arrays:", list(root.array_keys()))


In [ ]:
eq = xr.open_zarr(mapper, group="equilibrium", consolidated=False)

vars_ = list(eq.data_vars)
# quick filter for likely vertical-position signals
cands = [v for v in vars_ if any(k in v.lower() for k in ["z", "axis", "mag", "rm", "zm", "cent", "rmag", "zmag"])]
print("num vars:", len(vars_))
print("candidates:", cands[:80])


In [3]:
def mapper_fn(shot: int):
    base = f"s3://mast/level2/shots/{shot}.zarr"
    return fs.get_mapper(base)

# shots = list(range(30350, 30360))  # 10 shots only

# for shot in shots:
#     try:
#         eq = xr.open_zarr(mapper_fn(shot), group="equilibrium", consolidated=False)
#         print("OPEN OK", shot, "vars:", len(eq.data_vars))
#     except Exception as e:
#         print("OPEN FAIL", shot, type(e).__name__, str(e)[:200])

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
eq = xr.open_zarr(mapper, group="equilibrium", consolidated=False)

tname = [c for c in eq.coords if "time" in c.lower()][0]
t = eq[tname].values
z = eq["magnetic_axis_z"].values

plt.figure()
plt.plot(t, z)
plt.xlabel(tname)
plt.ylabel("magnetic_axis_z [m]")
plt.title("Magnetic axis vertical position")
plt.show()


In [ ]:
dt = 0.05        # 50 ms look-ahead
dz_thresh = 0.04 # 2 cm threshold
bn = eq["beta_normal"].values
idx = np.where(np.isfinite(bn) & (bn > 0.05))[0]
t0 = float(t[idx[0]]) if idx.size else float(t[len(t)//3])
t0
m = (t >= t0) & (t <= t0 + dt)
z0 = float(z[m][0])
unstable = float(np.max(np.abs(z[m] - z0)) > dz_thresh)
plt.figure()
plt.plot(t, z)
plt.axvline(t0, ls="--")
plt.axvline(t0+dt, ls="--")
plt.axhline(z0 + dz_thresh, ls="--")
plt.axhline(z0 - dz_thresh, ls="--")
plt.title(f"unstable={int(unstable)} over {dt*1e3:.0f} ms, thresh={dz_thresh*100:.1f} cm")
plt.show()
unstable


In [4]:
def extract_row_from_eq(eq, dt=0.05, dz_thresh=0.02, beta_thr=0.01):
    tname = [c for c in eq.coords if "time" in c.lower()][0]
    t = eq[tname].values

    # minimal required variables
    if "magnetic_axis_z" not in eq.data_vars or "beta_normal" not in eq.data_vars:
        return None

    z = eq["magnetic_axis_z"].values
    bn = eq["beta_normal"].values

    idx = np.where(np.isfinite(bn) & (bn > beta_thr))[0]
    if idx.size == 0:
        # fallback: first finite z
        idx = np.where(np.isfinite(z))[0]
        if idx.size == 0:
            return None
    t0 = float(t[idx[0]])

    m = (t >= t0) & (t <= t0 + dt)
    if np.sum(m) < 10:
        return None

    z0 = float(z[m][0])
    unstable = float(np.max(np.abs(z[m] - z0)) > dz_thresh)

    def at(v):
        return float(eq[v].sel({tname: t0}, method="nearest").values)

    # optional features (only keep those present)
    optional = [
        "beta_normal",
        "elongation_axis",
        "q_axis",
        "magnetic_axis_r",
        "lcfs_z",
        "x_point_z",
        "bphi_rmag",
        "bvac_rmag",
        "beta_tor_normal",  # keep if present
    ]
    row = {"t0": t0, "unstable": unstable}
    for v in optional:
        if v in eq.data_vars:
            row[v] = at(v)

    return row


In [5]:
# rows = []
# n_ok = 0
# n_fail = 0

# shots = list(range(30350, 30450))  # pick a band around your known good shot

# for shot in shots:
#     try:
#         eq = xr.open_zarr(mapper_fn(shot), group="equilibrium", consolidated=False)
#         row = extract_row_from_eq(eq)
#         if row is None:
#             n_fail += 1
#             continue
#         row["shot"] = shot
#         rows.append(row)
#         n_ok += 1
#     except Exception:
#         n_fail += 1

# df = pd.DataFrame(rows)
# print("ok:", n_ok, "fail:", n_fail)
# print("columns:", list(df.columns))
# df.head()


In [ ]:
shot = 30350
eq = xr.open_zarr(mapper_fn(shot), group="equilibrium", consolidated=False)

row = extract_row_from_eq(eq, debug=True)   # use your debug extractor
row


In [6]:
import pandas as pd
import xarray as xr

shots = list(range(30350, 30450))
rows = []

for shot in shots:
    try:
        eq = xr.open_zarr(mapper_fn(shot), group="equilibrium", consolidated=False)
        row = extract_row_from_eq(eq)
        if row is None:
            continue
        row["shot"] = shot
        rows.append(row)
    except Exception:
        continue

df = pd.DataFrame(rows)
print("rows:", len(df))
print("cols:", list(df.columns))
df.head()


rows: 0
cols: []


""
